# DISCOVERSE 专家数据、策略与三维渲染：完整实操

默认显示阶段命令，不运行实验。设置 `AMD_REPRO_STAGE` 为单元内的阶段编号后，只执行该阶段；配置源代码、数据、模型与解释器后依次完成数据准备、训练、评估与视频归档。当前阅读执行不代表这些计算已重新完成。


每个执行阶段均显式进入其工程目录，前一单元子进程中的目录和环境变量不向后隐式传递。认证操作请在终端完成后复用登录状态；本篇不自动启动交互登录。训练与评估的实际完成范围见对应结果与缺口章节。

In [1]:
from pathlib import Path
import json, os, sys
TOPIC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'code/amd_benchmarks').is_dir())
sys.path.insert(0, str(TOPIC / 'code'))
from amd_benchmarks.evidence import table
from amd_benchmarks.catalog import show_media, load_record, preview_stage
print('当前读取随课程保存的证据；未启动新的训练、仿真或系统维护。')


当前读取随课程保存的证据；未启动新的训练、仿真或系统维护。


# DISCOVERSE：ROCm 数据生成、训练与多视角视频

DISCOVERSE 是面向 Real2Sim2Real［真实到仿真再到真实］机器人学习的仿真框架，覆盖 AIRBOT、MMK2、专家状态机、模仿学习、LiDAR［激光雷达］与 3DGS［三维高斯泼溅］。本章从源码和模型下载开始，完成专家数据生成、LeRobot［机器人学习数据框架］转换、ACT/Diffusion Policy［扩散策略］训练、闭环评估和三路高清视频。

## 本章产出

- AIRBOT 与 MMK2 场景运行记录；
- 成功专家轨迹和逐回合生成清单；
- LeRobot 数据集、动作/观测映射和回放视频；
- ACT 或 Diffusion Policy 模型、闭环结果和多视角视频；
- 可选 3DGS 动态渲染。

安装后记录上游版本：



In [2]:
preview_stage('01-01', 'bash', 'git -C "$SRC_ROOT/DISCOVERSE" rev-parse HEAD\n', reference_only=False)


**阶段 `01-01`**

```bash
git -C "$SRC_ROOT/DISCOVERSE" rev-parse HEAD

```

未选择本阶段，不启动计算或下载。




## 1. 下载和安装



In [3]:
preview_stage('01-02', 'bash', 'cd "$SRC_ROOT"\ngit lfs install\ngit clone https://github.com/discoverse-dev/DISCOVERSE.git\ncd DISCOVERSE\n\nexport DISCOVERSE_ENV="$ENV_ROOT/discoverse-rocm"\npython3.10 -m venv "$DISCOVERSE_ENV"\nsource "$DISCOVERSE_ENV/bin/activate"\npython -m pip install --upgrade pip\npython -m pip install -e .\npython scripts/setup_submodules.py\npython scripts/check_installation.py\n', reference_only=False)


**阶段 `01-02`**

```bash
cd "$SRC_ROOT"
git lfs install
git clone https://github.com/discoverse-dev/DISCOVERSE.git
cd DISCOVERSE

export DISCOVERSE_ENV="$ENV_ROOT/discoverse-rocm"
python3.10 -m venv "$DISCOVERSE_ENV"
source "$DISCOVERSE_ENV/bin/activate"
python -m pip install --upgrade pip
python -m pip install -e .
python scripts/setup_submodules.py
python scripts/check_installation.py

```

未选择本阶段，不启动计算或下载。




按用途增加依赖：



In [4]:
preview_stage('01-03', 'bash', 'cd "$SRC_ROOT/DISCOVERSE"\npython -m pip install -e \'.[act_full]\'\npython -m pip install -e \'.[lidar,visualization]\'\npython -m pip install -e \'.[gs]\'\n', reference_only=False)


**阶段 `01-03`**

```bash
cd "$SRC_ROOT/DISCOVERSE"
python -m pip install -e '.[act_full]'
python -m pip install -e '.[lidar,visualization]'
python -m pip install -e '.[gs]'

```

未选择本阶段，不启动计算或下载。




3DGS［三维高斯泼溅］模型由 `tatp/DISCOVERSE-models` 提供，首次运行会自动下载，也可提前登录：



In [5]:
preview_stage('01-04', 'bash', 'cd "$SRC_ROOT/DISCOVERSE"\nhf auth login\n', reference_only=True)


**阶段 `01-04`**

```bash
cd "$SRC_ROOT/DISCOVERSE"
hf auth login

```

参考阶段：当前设备尚未完成完整训练验证，本单元不自动执行。




## 2. 基础运行检查



In [6]:
preview_stage('01-05', 'bash', 'cd "$SRC_ROOT/DISCOVERSE"\nexport MUJOCO_GL=egl\npython discoverse/robots_env/airbot_play_base.py\npython discoverse/robots_env/mmk2_base.py\n\npython examples/mocap_ik/mocap_ik_manipulator.py \\\n  -r airbot_play \\\n  -t place_block \\\n  --record \\\n  --camera-names global_camera wrist_camera\n', reference_only=False)


**阶段 `01-05`**

```bash
cd "$SRC_ROOT/DISCOVERSE"
export MUJOCO_GL=egl
python discoverse/robots_env/airbot_play_base.py
python discoverse/robots_env/mmk2_base.py

python examples/mocap_ik/mocap_ik_manipulator.py \
  -r airbot_play \
  -t place_block \
  --record \
  --camera-names global_camera wrist_camera

```

未选择本阶段，不启动计算或下载。




迁移验收覆盖 18 个运行检查、AIRBOT 12 个任务和 MMK2 8 个固定场景。基础检查通过后，再生成训练数据。

## 3. 专家轨迹生成

官方任务脚本已经包含专家状态机。以 AIRBOT 放置积木为例：



In [7]:
preview_stage('01-06', 'bash', 'cd "$SRC_ROOT/DISCOVERSE"\npython examples/tasks_airbot_play/place_block.py --help\n', reference_only=False)


**阶段 `01-06`**

```bash
cd "$SRC_ROOT/DISCOVERSE"
python examples/tasks_airbot_play/place_block.py --help

```

未选择本阶段，不启动计算或下载。




专家状态机需要同时满足运动学和物理约束。出现 IK［逆运动学］不可达时，先检查目标位姿是否超出机械臂工作空间、末端姿态是否导致关节限位，以及相邻状态之间的位姿跳变。对堆叠任务，将专家路径拆为预抓取、抓取、抬升、预放置、下降、释放和撤离，并对相邻状态做插值。支撑物体被碰倒时，调整接近方向和撤离方向，不通过增加质量替代路径修正。

不同版本的参数名可能变化，先以 `--help` 为准。批量生成时固定数据根目录、seed［随机种子］范围和目标成功数：



In [8]:
preview_stage('01-07', 'bash', 'cd "$SRC_ROOT/DISCOVERSE"\nexport DISCOVERSE_DATA_ROOT="$DATA_ROOT/discoverse"\nmkdir -p "$DATA_ROOT/discoverse/place_block"\n\nfor seed in $(seq 0 499); do\n  python examples/tasks_airbot_play/place_block.py \\\n    --data_idx "$seed" \\\n    --data_set_size 1 \\\n    --auto\ndone\n\nrsync -a data/place_block/ "$DATA_ROOT/discoverse/place_block/"\n', reference_only=False)


**阶段 `01-07`**

```bash
cd "$SRC_ROOT/DISCOVERSE"
export DISCOVERSE_DATA_ROOT="$DATA_ROOT/discoverse"
mkdir -p "$DATA_ROOT/discoverse/place_block"

for seed in $(seq 0 499); do
  python examples/tasks_airbot_play/place_block.py \
    --data_idx "$seed" \
    --data_set_size 1 \
    --auto
done

rsync -a data/place_block/ "$DATA_ROOT/discoverse/place_block/"

```

未选择本阶段，不启动计算或下载。




官方脚本先写入 DISCOVERSE 源码目录下的 `data/place_block`；最后一行把通过成功条件后实际保存的回合增量同步到大容量数据盘。再次执行相同 seed［随机种子］前，先检查目标回合是否已经存在，避免重复生成。

每条轨迹至少检查：

- 仿真成功条件为真；
- 状态、动作和相机帧数量一致；
- 时间戳严格递增；
- 末尾动作 padding［填充］不跨 episode［回合］；
- 回放仍能完成抓取、运输和放置。

我们归档的 `place_block` 专家语料为 500 个通过验收的 episode［回合］、44,439 个对齐帧。

## 4. 转换为训练数据

DISCOVERSE 原始数据通常包含 `obs_action.json`、相机图像或视频和任务元数据。转换为 LeRobot［机器人学习数据框架］时建立统一字段：

```text
observation.state
observation.images.{camera}
action
task
episode_index
frame_index
timestamp
```

先对单条轨迹转换，再批量执行：



In [9]:
preview_stage('01-08', 'bash', 'cd "$SRC_ROOT/DISCOVERSE"\npython "$EVERY_EMBODIED_ROOT/16-专题组队学习/04-AMD-ROCm策略复刻专题/code/convert_discoverse_to_lerobot.py" \\\n  --input-root "$DATA_ROOT/discoverse/place_block" \\\n  --output-root "$DATA_ROOT/discoverse-lerobot/place_block" \\\n  --repo-id local/discoverse-place-block \\\n  --task \'Place the green block in the pink bowl\' \\\n  --fps 20 \\\n  --image-size 256\n', reference_only=False)


**阶段 `01-08`**

```bash
cd "$SRC_ROOT/DISCOVERSE"
python "$EVERY_EMBODIED_ROOT/16-专题组队学习/04-AMD-ROCm策略复刻专题/code/convert_discoverse_to_lerobot.py" \
  --input-root "$DATA_ROOT/discoverse/place_block" \
  --output-root "$DATA_ROOT/discoverse-lerobot/place_block" \
  --repo-id local/discoverse-place-block \
  --task 'Place the green block in the pink bowl' \
  --fps 20 \
  --image-size 256

```

未选择本阶段，不启动计算或下载。




转换器读取每个成功回合的 `obs_action.json`、`cam_0.mp4` 和 `cam_1.mp4`，校验两路图像、7 维关节状态与 7 维动作逐帧等长。DISCOVERSE 在执行动作后记录观测，因此默认 `--action-bias=-1`，把观测时刻与下一条动作对齐。转换结果中的 `discoverse_conversion_manifest.json` 记录源回合、帧数和动作索引范围。不要仅比较数组 shape［形状］，还要回放动作验证时序。

## 5. ACT 训练



In [10]:
preview_stage('01-09', 'bash', 'cd "$SRC_ROOT/DISCOVERSE"\nlerobot-train \\\n  --dataset.repo_id=local/discoverse-place-block \\\n  --dataset.root="$DATA_ROOT/discoverse-lerobot/place_block" \\\n  --policy.type=act \\\n  --policy.device=cuda \\\n  --policy.chunk_size=100 \\\n  --policy.n_action_steps=100 \\\n  --steps=100000 \\\n  --batch_size=32 \\\n  --save_freq=5000 \\\n  --log_freq=100 \\\n  --wandb.enable=false \\\n  --output_dir="$RUN_ROOT/discoverse/act_place_block"\n', reference_only=False)


**阶段 `01-09`**

```bash
cd "$SRC_ROOT/DISCOVERSE"
lerobot-train \
  --dataset.repo_id=local/discoverse-place-block \
  --dataset.root="$DATA_ROOT/discoverse-lerobot/place_block" \
  --policy.type=act \
  --policy.device=cuda \
  --policy.chunk_size=100 \
  --policy.n_action_steps=100 \
  --steps=100000 \
  --batch_size=32 \
  --save_freq=5000 \
  --log_freq=100 \
  --wandb.enable=false \
  --output_dir="$RUN_ROOT/discoverse/act_place_block"

```

未选择本阶段，不启动计算或下载。




在 ROCm［AMD 开放计算平台］上 `--policy.device=cuda` 仍是正确写法。先用 10–50 steps［训练步数］验证数据与反向传播，再启动正式长训。

## 6. Diffusion Policy 训练



In [11]:
preview_stage('01-10', 'bash', 'cd "$SRC_ROOT/DISCOVERSE"\nlerobot-train \\\n  --dataset.repo_id=local/discoverse-place-block \\\n  --dataset.root="$DATA_ROOT/discoverse-lerobot/place_block" \\\n  --policy.type=diffusion \\\n  --policy.device=cuda \\\n  --steps=100000 \\\n  --batch_size=16 \\\n  --save_freq=5000 \\\n  --log_freq=100 \\\n  --wandb.enable=false \\\n  --output_dir="$RUN_ROOT/discoverse/diffusion_place_block"\n', reference_only=False)


**阶段 `01-10`**

```bash
cd "$SRC_ROOT/DISCOVERSE"
lerobot-train \
  --dataset.repo_id=local/discoverse-place-block \
  --dataset.root="$DATA_ROOT/discoverse-lerobot/place_block" \
  --policy.type=diffusion \
  --policy.device=cuda \
  --steps=100000 \
  --batch_size=16 \
  --save_freq=5000 \
  --log_freq=100 \
  --wandb.enable=false \
  --output_dir="$RUN_ROOT/discoverse/diffusion_place_block"

```

未选择本阶段，不启动计算或下载。




具体策略名称以所安装 LeRobot［机器人学习数据框架］版本的 `lerobot-train --help` 为准。训练配置必须保存观测映射、动作归一化和时间窗口。

## 7. 闭环评估

评估循环必须让策略动作真实推进模拟器：



此单元是通用接口示意，尚缺与上面通用训练权重匹配的部署适配器。实际原生策略部署、旧完整训练和闭环结果请看 [历史完整训练与闭环诊断](02_历史完整训练与闭环诊断.ipynb)。

In [12]:
preview_stage('01-11', 'python', 'observation = env.reset()\nfor step in range(max_steps):\n    action = policy.select_action(observation)\n    observation, reward, terminated, truncated, info = env.step(action)\n    recorder.append(observation, action, info)\n    if terminated or truncated:\n        break\n', reference_only=True)


**阶段 `01-11`**

```python
observation = env.reset()
for step in range(max_steps):
    action = policy.select_action(observation)
    observation, reward, terminated, truncated, info = env.step(action)
    recorder.append(observation, action, info)
    if terminated or truncated:
        break

```

参考阶段：当前设备尚未完成完整训练验证，本单元不自动执行。




固定 train-like［训练分布相似］与 unseen［未见分布］两组 seed［随机种子］，分别统计接近、抓取、抬升、运输、对齐和最终放置。评估结果与视频写入同一回合目录。

## 8. 多视角高清输出

MMK2 的 `kiwi_pick.py` 默认已经记录 3 个相机，但分辨率为 640×480。复制任务脚本并只调整录像分辨率，可以保留官方专家状态机：



In [13]:
preview_stage('01-12', 'bash', 'cd "$SRC_ROOT/DISCOVERSE"\ncd "$SRC_ROOT/DISCOVERSE"\nmkdir -p "$RUN_ROOT/discoverse/mmk2_kiwi_hd"\ncp examples/tasks_mmk2/kiwi_pick.py \\\n  "$RUN_ROOT/discoverse/mmk2_kiwi_hd/kiwi_pick_hd.py"\n\npython - <<\'PY\'\nimport os\nfrom pathlib import Path\n\npath = Path(os.environ["RUN_ROOT"]) / "discoverse/mmk2_kiwi_hd/kiwi_pick_hd.py"\ntext = path.read_text()\ntext = text.replace(\'"width"  : 640\', \'"width"  : 1920\')\ntext = text.replace(\'"height" : 480\', \'"height" : 1080\')\npath.write_text(text)\nPY\n\npython "$RUN_ROOT/discoverse/mmk2_kiwi_hd/kiwi_pick_hd.py" \\\n  --data_idx 0 \\\n  --data_set_size 1 \\\n  --auto\n', reference_only=False)


**阶段 `01-12`**

```bash
cd "$SRC_ROOT/DISCOVERSE"
cd "$SRC_ROOT/DISCOVERSE"
mkdir -p "$RUN_ROOT/discoverse/mmk2_kiwi_hd"
cp examples/tasks_mmk2/kiwi_pick.py \
  "$RUN_ROOT/discoverse/mmk2_kiwi_hd/kiwi_pick_hd.py"

python - <<'PY'
import os
from pathlib import Path

path = Path(os.environ["RUN_ROOT"]) / "discoverse/mmk2_kiwi_hd/kiwi_pick_hd.py"
text = path.read_text()
text = text.replace('"width"  : 640', '"width"  : 1920')
text = text.replace('"height" : 480', '"height" : 1080')
path.write_text(text)
PY

python "$RUN_ROOT/discoverse/mmk2_kiwi_hd/kiwi_pick_hd.py" \
  --data_idx 0 \
  --data_set_size 1 \
  --auto

```

未选择本阶段，不启动计算或下载。




任务成功后，三路原片位于 `data/pick_kiwi/000/cam_0.mp4`、`cam_1.mp4` 和 `cam_2.mp4`。脚本只在成功条件成立后写盘，因此这些视频与对应的专家动作属于同一回合。

保持单相机原片，同时生成三视角合成版：



In [14]:
preview_stage('01-13', 'bash', 'cd "$SRC_ROOT/DISCOVERSE"\nffmpeg \\\n  -i data/pick_kiwi/000/cam_0.mp4 \\\n  -i data/pick_kiwi/000/cam_1.mp4 \\\n  -i data/pick_kiwi/000/cam_2.mp4 \\\n  -filter_complex \'[0:v][1:v][2:v]hstack=inputs=3[v]\' \\\n  -map \'[v]\' -c:v libx264 -crf 18 -pix_fmt yuv420p \\\n  "$RUN_ROOT/discoverse/mmk2_kiwi_hd/composite.mp4"\n', reference_only=False)


**阶段 `01-13`**

```bash
cd "$SRC_ROOT/DISCOVERSE"
ffmpeg \
  -i data/pick_kiwi/000/cam_0.mp4 \
  -i data/pick_kiwi/000/cam_1.mp4 \
  -i data/pick_kiwi/000/cam_2.mp4 \
  -filter_complex '[0:v][1:v][2:v]hstack=inputs=3[v]' \
  -map '[v]' -c:v libx264 -crf 18 -pix_fmt yuv420p \
  "$RUN_ROOT/discoverse/mmk2_kiwi_hd/composite.mp4"

```

未选择本阶段，不启动计算或下载。




## 9. 3DGS 渲染



In [15]:
preview_stage('01-14', 'bash', 'cd "$SRC_ROOT/DISCOVERSE"\npython -m pip install -e \'.[gs]\'\nhf auth login\n', reference_only=True)


**阶段 `01-14`**

```bash
cd "$SRC_ROOT/DISCOVERSE"
python -m pip install -e '.[gs]'
hf auth login

```

参考阶段：当前设备尚未完成完整训练验证，本单元不自动执行。




在支持 Gaussian renderer［高斯渲染器］的场景中设置 `cfg.use_gaussian_renderer = True`，交互窗口用 `Ctrl+g` 切换。验证时保存：

- 实际加载的 `.ply` 点云路径；
- 点数量与渲染分辨率；
- 至少一段动态回放；
- 同任务 MuJoCo 网格渲染作为对照。

## 10. AMD 迁移重点

| 模块 | 迁移内容 |
| --- | --- |
| PyTorch［张量计算框架］策略 | 保留 `torch.cuda` 接口，安装 ROCm wheel［AMD 软件包］ |
| MuJoCo | 使用 EGL［无窗口图形渲染接口］输出视频 |
| 3DGS［三维高斯泼溅］ | 替换 CUDA 专用扩展或使用 ROCm 可编译实现 |
| 数据 | 修正 episode［回合］边界、动作 chunk［动作块］和相机时序 |
| ACT | 保持训练与执行的 chunk［动作块］长度一致 |
| 多机器人 | 分别验证机器人 MJCF、控制器和成功条件 |

## 11. 结果口径

环境运行、专家成功率和学习策略成功率分别记录。示例迁移记录包括运行检查 `18/18`、AIRBOT `12/12`、MMK2 `8/8`、高清任务 `4/4` 和 500 条专家轨迹；训练模型使用自己的闭环任务和分母汇总。

## 12. 官方资料

- [DISCOVERSE 源码](https://github.com/discoverse-dev/DISCOVERSE)
- [DISCOVERSE 项目主页](https://air-discoverse.github.io/)
- [DISCOVERSE 模型](https://huggingface.co/tatp/DISCOVERSE-models)


## 原工程的完整动作分块训练实现

下面直接展开原工程 `policies/act/policy_train.py`：模型与优化器构建、数据加载、训练／验证、最佳权重与定期保存。源码从已完成的公开归档按完整指纹恢复；用于理解与复现原工程训练接口。历史闭环成绩和视频仍按前文的独立配方记录，不能混用。

完整运行前，设置 `DISCOVERSE_SOURCE_ROOT` 为已恢复的原工程目录、选择其既有训练环境，并配置原工程任务参数；默认仅阅读，不启动训练。

In [16]:
RUN_FULL = os.environ.get("AMD_RUN_FULL", "0") == "1"
if RUN_FULL:
    _discoverse_source = Path(os.environ["DISCOVERSE_SOURCE_ROOT"]).resolve()
    assert (_discoverse_source / "policies/act").is_dir()
    sys.path.insert(0, str(_discoverse_source))


In [17]:
if RUN_FULL:
    import argparse


In [18]:
if RUN_FULL:
    import logging


In [19]:
if RUN_FULL:
    import os


In [20]:
if RUN_FULL:
    import pickle


In [21]:
if RUN_FULL:
    import shutil


In [22]:
if RUN_FULL:
    import time


In [23]:
if RUN_FULL:
    from copy import deepcopy


In [24]:
if RUN_FULL:
    import matplotlib.pyplot as plt


In [25]:
if RUN_FULL:
    import numpy as np


In [26]:
if RUN_FULL:
    import torch


In [27]:
if RUN_FULL:
    from tqdm import tqdm


In [28]:
if RUN_FULL:
    from configurations.task_configs.config_tools.basic_configer import (
        basic_parser,
        get_all_config,
    )


In [29]:
if RUN_FULL:
    from policies.common.maker import make_policy


In [30]:
if RUN_FULL:
    from utils.utils import (
        GPUer,
        LoadDataConfig,
        compute_dict_mean,
        detach_dict,
        load_data,
        set_seed,
    )


In [31]:
if RUN_FULL:
    logging.basicConfig(level=logging.INFO)


In [32]:
if RUN_FULL:
    logger = logging.getLogger(__name__)


In [33]:
if RUN_FULL:
    def main(args: dict):
        # get all config
        all_config = get_all_config(args, "train")
        ckpt_dir = all_config["ckpt_dir"]
        stats_path = all_config["stats_path"]
        gpu_threshold = all_config.get("gpu_threshold", 10)

        # 加载数据及统计信息
        train_dataloader, val_dataloader, stats = load_data(
            LoadDataConfig(
                **all_config["load_data"],
                camera_names=all_config["camera_names"],
                chunk_sizes={"action": all_config["policy_config"]["chunk_size"]},
            )
        )
        # 创建保存路径
        os.makedirs(ckpt_dir, exist_ok=True)
        stats_dir = os.path.dirname(stats_path)
        os.makedirs(stats_dir, exist_ok=True)
        # 复制配置文件到stats_dir
        shutil.copy(all_config["config_file_sys_path"], stats_dir)
        # 保存统计数据
        stats_path = (
            os.path.join(ckpt_dir, "dataset_stats.pkl") if stats_path == "" else stats_path
        )
        print(f"Saving stats into {stats_path}...")
        with open(stats_path, "wb") as f:
            pickle.dump(stats, f)
        # 保存关键信息（must pop functions)
        key_info_path = os.path.join(stats_dir, "key_info.pkl")
        print(f"Saving key info into {key_info_path}...")
        all_config_cp = deepcopy(all_config)
        all_config_cp["policy_config"].pop("policy_maker")
        all_config_cp["environments"].pop("environment_maker")
        all_config_cp["load_data"].pop("augmentors")
        key_info = {
            "init_info": {
                "init_joint": all_config_cp["start_joint"],
                "init_action": all_config_cp["start_action"],
            },
            "all_config": all_config_cp,
        }
        with open(key_info_path, "wb") as f:
            pickle.dump(key_info, f)
        # wait for free GPU
        target_gpus = os.environ.get("CUDA_VISIBLE_DEVICES", "")
        target_gpus = target_gpus.split(",")
        target_gpus = [int(gpu) for gpu in target_gpus if gpu != ""]
        waiting_time = 60
        while True:
            free_gpus, gpus_num, gpu_utilizations = GPUer.check_all_gpus_idle(gpu_threshold)
            if len(free_gpus) > 0:
                if len(target_gpus) > 0:
                    not_free_ids = []
                    for index, target_gpu in enumerate(target_gpus):
                        if target_gpu >= gpus_num:
                            raise ValueError(
                                f"Target GPU id (from 0) {target_gpu} is not valid, only {gpus_num} gpus available"
                            )
                        elif target_gpu not in free_gpus:
                            not_free_ids.append(index)
                    if len(not_free_ids) != 0:
                        print(
                            f"Target GPU {target_gpus[not_free_ids]} is not free ({gpu_utilizations[not_free_ids]}), waiting for {waiting_time} senconds..."
                        )
                        time.sleep(waiting_time)
                        continue
                else:
                    free_gpu = free_gpus[0]
                    os.environ["CUDA_VISIBLE_DEVICES"] = str(free_gpu)
                    print(f"Using GPU {free_gpu}")
                    target_gpus = [int(free_gpu)]
                break
            else:
                print(f"No free GPU, waiting for {waiting_time} senconds...")
                time.sleep(waiting_time)
        print(f"Using GPU: {target_gpus}")
        all_config["target_gpus"] = target_gpus
        # train policy
        set_seed(all_config["seed"])
        best_ckpt_info = train_bc(train_dataloader, val_dataloader, all_config)
        best_epoch, min_val_loss, best_state_dict = best_ckpt_info
        print(f"Best ckpt, val loss {min_val_loss:.6f} @ epoch{best_epoch}")


In [34]:
if RUN_FULL:
    def make_optimizer(policy):
        if hasattr(policy, "configure_optimizers"):
            optimizer = policy.configure_optimizers()
        else:
            # TODO: 默认使用Adam优化器
            print("Warning: Using default optimizer")
        return optimizer


In [35]:
if RUN_FULL:
    def forward_pass(data: torch.Tensor, policy):
        image_data, qpos_data, action_data, is_pad = data
        image_data, qpos_data, action_data, is_pad = (
            image_data.cuda(),
            qpos_data.cuda(),
            action_data.cuda(),
            is_pad.cuda(),
        )
        # print("image_data.shape:", image_data.shape)
        return policy(qpos_data, image_data, action_data, is_pad)  # TODO remove None


In [36]:
if RUN_FULL:
    def get_epoch_base(pretrain_path, epoch_base):
        if pretrain_path == "":
            epoch_base = 0
        elif isinstance(epoch_base, str) and epoch_base.lstrip("-").isdigit():
            epoch_base = int(epoch_base)
        elif epoch_base == "AUTO":
            if (
                pretrain_path in ["best", "last"]
                or "best" in pretrain_path
                or "last" in pretrain_path
            ):
                epoch_base = 0
            else:
                try:
                    epoch_base = int(pretrain_path.split("_")[-3])
                except:
                    try:
                        epoch_base = int(pretrain_path.split("_")[-2])
                    except:
                        raise ValueError(
                            f"Invalid pretrain_ckpt_path to auto get epoch bias: {pretrain_path}"
                        )
        return epoch_base


In [37]:
if RUN_FULL:
    def train_bc(train_dataloader, val_dataloader, config):
        num_epochs = config["num_epochs"]
        ckpt_dir = config["ckpt_dir"]
        seed = config["seed"]
        policy_config = config["policy_config"]
        stats_dir = os.path.dirname(config["stats_path"])
        eval_every = 3.14 if config["eval_every"] == 0 else config["eval_every"]
        parallel = config["parallel"]
        if config["eval_every"] != 0:
            from policy_evaluate import eval_bc

        # make policy
        pretrain_path = config["pretrain_ckpt_path"]
        policy_config["ckpt_path"] = pretrain_path
        policy = make_policy(policy_config, "train")

        # get epoch base
        epoch_base = get_epoch_base(pretrain_path, config["pretrain_epoch_base"])

        # make optimizer
        optimizer = make_optimizer(policy)

        # set GPU device
        if parallel is not None:
            if parallel["mode"] == "DP":
                if parallel.get("device_ids", None) is not None:
                    device_ids = parallel["device_ids"]
                else:
                    device_ids = config["target_gpus"]
                assert len(device_ids) > 1, "DP mode requires more than 1 GPU"
                print(f"Using GPUs {device_ids} for DataParallel training")
                device_ids = list(range(len(device_ids)))
                policy = torch.nn.DataParallel(policy, device_ids=device_ids)
                optimizer = torch.nn.DataParallel(optimizer, device_ids=device_ids)
            elif parallel["mode"] == "DDP":
                # TODO: can not use DDP for now
                raise NotImplementedError
                policy = torch.nn.parallel.DistributedDataParallel(
                    policy, device_ids=parallel["device_ids"]
                )
            else:
                raise ValueError(f'Invalid parallel mode: {parallel["mode"]}')

        # training loop
        train_history = []
        validation_history = []
        min_val_loss = np.inf
        best_ckpt_info = None
        for epoch in tqdm(range(epoch_base, num_epochs + epoch_base)):
            print(f"\nEpoch {epoch}")
            step = epoch - epoch_base + 1
            # validation
            if step % config["validate_every"] == 0:
                print("validating")
                with torch.inference_mode():
                    policy.eval()  # this is useless?
                    epoch_dicts = []
                    for batch_idx, data in enumerate(val_dataloader):
                        forward_dict = forward_pass(data, policy)
                        epoch_dicts.append(forward_dict)
                    epoch_summary = compute_dict_mean(epoch_dicts)
                    validation_history.append(epoch_summary)

                    epoch_val_loss = epoch_summary["loss"]
                    if epoch_val_loss < min_val_loss:
                        min_val_loss = epoch_val_loss
                        best_ckpt_info = (
                            epoch,
                            min_val_loss,
                            deepcopy(policy.state_dict()),
                        )
                print(f"Val loss:   {epoch_val_loss:.5f}")
                summary_string = ""
                for k, v in epoch_summary.items():
                    summary_string += f"{k}: {v.item():.3f} "
                print(summary_string)

            # evaluation #TODO: 目前不支持训练过程中的评估
            if step % eval_every == 0:
                # first save then eval
                ckpt_name = f"policy_step_{step}_seed_{seed}.ckpt"
                ckpt_path = os.path.join(ckpt_dir, ckpt_name)
                torch.save(policy.state_dict(), ckpt_path)
                success, _ = eval_bc(config, ckpt_name, save_episode=True, num_rollouts=10)

            # training
            policy.train()
            optimizer.zero_grad()
            for batch_idx, data in enumerate(train_dataloader):
                forward_dict = forward_pass(data, policy)
                # backward
                loss = forward_dict["loss"]
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()
                train_history.append(detach_dict(forward_dict))
            epoch_summary = compute_dict_mean(
                train_history[
                    (batch_idx + 1)
                    * (epoch - epoch_base) : (batch_idx + 1)
                    * (epoch - epoch_base + 1)
                ]
            )
            epoch_train_loss = epoch_summary["loss"]
            print(f"Train loss: {epoch_train_loss:.5f}")
            summary_string = ""
            for k, v in epoch_summary.items():
                summary_string += f"{k}: {v.item():.3f} "
            print(summary_string)

            if step % config["save_every"] == 0:
                ckpt_path = os.path.join(ckpt_dir, f"policy_epoch_{epoch}_seed_{seed}.ckpt")
                torch.save(policy.state_dict(), ckpt_path)
                plot_history(train_history, validation_history, epoch, stats_dir, seed)

        assert (
            best_ckpt_info is not None
        ), "No best ckpt found, make sure validate_every <= num_epochs"
        # training finished
        # save last and best ckpts
        ckpt_path = os.path.join(stats_dir, f"policy_last.ckpt")
        torch.save(policy.state_dict(), ckpt_path)
        best_epoch, min_val_loss, best_state_dict = best_ckpt_info
        policy_best_path = os.path.join(stats_dir, f"policy_best.ckpt")
        torch.save(best_state_dict, policy_best_path)

        # save training curves
        plot_history(train_history, validation_history, num_epochs, stats_dir, seed)

        # show and save result description
        description = (
            f"Best ckpt: val loss {min_val_loss:.6f} @ epoch{best_epoch} with seed {seed}"
        )
        print("Training finished.")
        print(description)
        with open(os.path.join(stats_dir, "description.txt"), "w") as f:
            f.write(description)
        return best_ckpt_info


In [38]:
if RUN_FULL:
    def plot_history(train_history, validation_history, num_epochs, stats_dir, seed):
        """save training curves to stats_dir"""
        for key in train_history[0]:
            plot_path = os.path.join(stats_dir, f"train_val_{key}_seed_{seed}.png")
            plt.figure()
            train_values = [summary[key].item() for summary in train_history]
            val_values = [summary[key].item() for summary in validation_history]
            plt.plot(
                np.linspace(0, num_epochs - 1, len(train_history)),
                train_values,
                label="train",
            )
            plt.plot(
                np.linspace(0, num_epochs - 1, len(validation_history)),
                val_values,
                label="validation",
            )
            # plt.ylim([-0.1, 1])
            plt.tight_layout()
            plt.legend()
            plt.title(key)
            plt.savefig(plot_path)
            plt.close()
        print(f"Saved plots to {stats_dir}")


In [39]:
if RUN_FULL:
    def parser_train(parser: argparse.ArgumentParser = None):
        if parser is None:
            parser = basic_parser()
        parser.add_argument(
            "-bs",
            "--batch_size",
            action="store",
            type=int,
            help="batch_size",
            required=False,
        )
        parser.add_argument(
            "-lr",
            "--learning_rate",
            action="store",
            type=float,
            help="learning_rate",
            required=False,
        )
        parser.add_argument(
            "-ne",
            "--num_epochs",
            action="store",
            type=int,
            help="num_epochs",
            required=False,
        )
        parser.add_argument(
            "-pcp",
            "--pretrain_ckpt_path",
            action="store",
            type=str,
            help="pretrain_ckpt_path",
            required=False,
        )
        parser.add_argument(
            "-peb",
            "--pretrain_epoch_base",
            action="store",
            type=str,
            help="pretrain_epoch_base",
            required=False,
        )
        parser.add_argument(
            "-ee", "--eval_every", action="store", type=int, help="eval_every"
        )
        parser.add_argument(
            "-ve",
            "--validate_every",
            action="store",
            type=int,
            help="validate_every",
            required=False,
        )
        parser.add_argument(
            "-se",
            "--save_every",
            action="store",
            type=int,
            help="save_every",
            required=False,
        )
        parser.add_argument(
            "-smd",
            "--skip_mirrored_data",
            action="store",
            type=bool,
            help="skip_mirrored_data",
            required=False,
        )
        parser.add_argument(
            "-gth",
            "--gpu_threshold",
            action="store",
            type=float,
            help="gpu_threshold",
            required=False,
        )
        # set time_stamp  # TODO: used to load pretrain model
        parser.add_argument(
            "-ts",
            "--time_stamp",
            action="store",
            type=str,
            help="time_stamp",
            required=False,
        )
        # parser.add_argument('--actuator_network_dir', action='store', type=str, help='actuator_network_dir', required=False)
        # parser.add_argument('--history_len', action='store', type=int)
        # parser.add_argument('--future_len', action='store', type=int)
        # parser.add_argument('--prediction_len', action='store', type=int)
        return parser


In [40]:
if RUN_FULL:
    import shlex
    _parser = basic_parser()
    parser_train(_parser)
    _arguments = _parser.parse_args(shlex.split(os.environ["DISCOVERSE_TRAIN_ARGS"]))
    main(vars(_arguments))
else:
    print("完整训练关闭；已展开原工程训练与验证逻辑。历史评估结果和视频保留。")


完整训练关闭；已展开原工程训练与验证逻辑。历史评估结果和视频保留。
